In [ ]:
from pulses import Tukey
from sequence import Sequence, Modulation
from system import System
from simulator import Simulate
from scipy.io import savemat
from numpy import ndarray
from json import load
# or
from yaml import safe_load
from typing import Callable
from pulses import Pulse
from numpy import ndarray, array, empty, diag, fliplr, sum, zeros, kron, eye, pi, sqrt, exp, sin, cos
from scipy.integrate import quad
from scipy.linalg import block_diag

In [2]:
pulse = Tukey(duration = 0.001, shape = 0.3, flipAngle = 200, offset = 7000)

In [3]:
sequence = Sequence(modulation = Modulation.CM, pulse = pulse, N_pulsePerOffset = 1, N_pulse = 4, N_burst = 10, N_adc = 80, dt_interPulse = 0.0015,
                        TR_burst = 0.1, dt_lastBurst = 0.001, ES = 0.006, TR = 3, readout_flipAngle = 5)

In [4]:
system = System(M0a = 1, M0b = 0.1, T1f = 1, T1b = 1, T1D = 0.01, T2f = 0.1, T2b = 1e-5, R = 10)

In [7]:
MT0, MTs, *MTds = Simulate(system, sequence)

(5, 5)
(1, 5)


ValueError: operands could not be broadcast together with shapes (0,) (5,5) 

In [5]:
from system import System
from sequence import Sequence, Modulation
from numpy import ndarray, zeros, kron, eye, diag, array, sum, vstack, hstack, abs, matmul, radians, cos
from numpy.linalg import matrix_power
from scipy.linalg import expm, block_diag, eig

In [8]:
    HomogenizeCol: ndarray = zeros((1, 1 + 2* system.N_compartments))
    HomogenizeCol[0] = system.poolFree_M0 / system.poolFree_T1
    HomogenizeCol[1::2] = system.poolBound_M0 / system.poolBound_T1

    REX = block_diag(
        -(1. / system.poolFree_T1 + system.poolFreeBound_exchangeRate * sum(array(system.poolBound_M0))),
        kron(
            eye(system.N_compartments),
            diag( [ -(1. / system.poolBound_T1 + system.poolFreeBound_exchangeRate * system.poolFree_M0), 0 ] )
        )
    )
    
    REX[1::2, 0] = system.poolFreeBound_exchangeRate * system.poolBound_M0
    REX[0, 1::2] = system.poolFreeBound_exchangeRate * system.poolFree_M0

    # Assuming only 1 free pool with 1 single compartment, filling the dipolar compartment relaxations
    REX[2::2, 2::2] = diag( array( [-1. / system.poolBound_T1D] ).flatten() )

In [11]:
print(REX.shape)
print(HomogenizeCol.shape)
toto = zeros((1, 2 + 2 * system.N_compartments))
print(toto.shape)

(5, 5)
(1, 5)
(1, 6)


In [31]:
totou = hstack([REX, HomogenizeCol.T])
print(totou.shape)

(5, 6)


In [9]:
    mat_REX = vstack([hstack([REX, HomogenizeCol.T]), zeros((1, 2 + 2 * system.N_compartments))])


In [ ]:
    mat_REX = vstack([hstack([REX, HomogenizeCol.T]), zeros((1, 2 + 2 * system.N_compartments))])

    evol_relax_interPulse = expm(mat_REX * (sequence.dt_interPulse - sequence.pulse.duration))
    evol_relax_interReadRF = expm(mat_REX * sequence.ES)
    evol_relax_recovery = expm(mat_REX * sequence.duration_recovery)
    evol_relax_TR_burst = expm(mat_REX * (sequence.TR_burst - sequence.N_pulse * sequence.dt_interPulse))
    evol_relax_lastBurst = expm(mat_REX * (sequence.dt_LastBurst - sequence.N_pulse * sequence.dt_interPulse))
    evol_relax_fullPrep = expm(mat_REX * sequence.duration_preparation)

    evol_rf_readoutInstantAction = eye(2 + 2 * system.N_compartments)
    evol_rf_readoutInstantAction[0, 0] = cos(radians(sequence.readout_flipAngle))   



In [ ]:
system.poolBound_Rrf_singleSat_Positive

In [15]:
system.poolBound_Rrf_singleSat_Positive + system.poolFree_Rrf + REX

ValueError: operands could not be broadcast together with shapes (0,) (5,5) 

In [18]:
aaa=system.poolBound_Rrf_singleSat_Positive
print(aaa)
print(system.poolFree_Rrf)

[]
[]


In [ ]:
    evol_rf_singleSat_Positive = expm( 
                                      vstack( 
                                              hstack( [ system.poolBound_Rrf_singleSat_Positive + system.poolFree_Rrf + REX, HomogenizeCol.T ] ), 
                                              zeros( (1, 2 + 2 * system.N_compartments) ) 
                                      ) * sequence.pulse.duration
                                    )

ValueError: operands could not be broadcast together with shapes (0,) (5,5) 

In [8]:
block_diag(sum(array(4)), kron(eye(2), diag([2, 0])))


array([[4., 0., 0., 0., 0.],
       [0., 2., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 2., 0.],
       [0., 0., 0., 0., 0.]])

In [11]:
system.N_compartments

2